**Chain for Agent** : 
It is a most important building block of the Langchain and it ties up with the llm with a prompt to carry out the sequence of operations on text and other data.


**Types** : 
* **Simple Sequential Chain** : It is another type of chain. The idea is to combine multiple chains where the output of one chain is the input of the next chain.
* **Sequential Chain** : Used when steps require multiple inputs or produce multiple outputs. It uses explicit input and output variable names so data can pass flexibly between non-adjacent steps or carry over across the entire pipeline.Example: Take a product description and customer review $\rightarrow$ translate review $\rightarrow$ write a summary $\rightarrow$ generate a response using both the summary and original review language.
* **Router Chain** : A Router Chain dynamically routes an incoming prompt or input to one of several sub-chains based on context, intent, or topic. Instead of processing data through every step linearly, it acts as a decision-making gateway to choose the single best specialized chain for the task.

**Router Chain** (e.g., LLMRouterChain): Uses an LLM to analyze the user input and decide which specific sub-chain to trigger.

**Destination Chains**: A collection of specialized chains designed for specific tasks (e.g., a Math Chain, a History Chain, a Coding Chain).

**Default Chain**: A fallback chain triggered if the router cannot determine a clear match among the destination options.


In [ ]:
import os
import openai
from langchain.chat_models import ChatOpenAI

In [ ]:
client = openai.OpenAI(api_keys=, base_url=) #The latest function to call the OpenAI LLM

In [ ]:
openai_api_keys = os.environ['OPENAI_API_KEYS'] #Load the openai api keys or use the alternative open router platform to get free api keys

In [ ]:
llm_model = "gpt-3.5-turbo" #Load the LLM Model or any openai model that is available

In [ ]:
from langchain.prompts import ChatPromptTemplate #Output specifier
from langchain.chains import LLMChain #To perform the sequence of the operation, we use LLM Chain

In [ ]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes a {product} ?"
)

In [ ]:
chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
product = "Queen Size Sheet Set"
chain.run(prompt)

**Output** : 'Regal Linens Co.'

## SimpleSequentialChain

In [ ]:
from langchain.chains import SimpleSequentialChain

In [ ]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [ ]:
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [ ]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [ ]:
overall_chain = SimpleSequentialChain(chains=[chain_one, chain_two], verbose=True)

In [ ]:
overall_chain.run(product)

**Output** : 'Regal Linens Co. offers luxurious and high-quality linens for bedrooms and bathrooms, providing elegance and comfort for your home.'

## SequentialChain

This sequential chain for the complex tasking for a LLM Model

In [ ]:
from langchain.chains import SequentialChain

In [ ]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [ ]:
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )


In [ ]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )


In [ ]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [ ]:

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [ ]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)

In [ ]:
review = df.Review[5]
overall_chain(review)

**Output** : 
{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': '"I find the taste mediocre. The foam doesn\'t last, it\'s strange. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?"',
 'summary': 'The reviewer found the taste of the product to be mediocre, with the foam not lasting, leading them to suspect the product may be an old batch or counterfeit.',
 'followup_message': "Merci pour votre avis honnête sur le produit. Il est regrettable d'apprendre que la saveur n'était que moyenne et que la mousse ne durait pas. Votre soupçon selon lequel il pourrait s'agir d'un lot ancien ou contrefait est compréhensible. Il est important de toujours acheter des produits authentiques pour garantir une expérience de qualité. Peut-être pourriez-vous contacter le fabricant pour obtenir plus d'informations sur la provenance du produit. Merci encore pour vos commentaires utiles."}

## Router Chain

Router Chains are used for a complex task where there are sub chains with different subjects to work upon with Multiple chains for information dividing according the subchain

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [ ]:
prompt_infos = [ #This is the 4 subchains for every prompt that user will ask out for the LLM Model.
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [ ]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.prompt import PromptTemplate

In [ ]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [ ]:

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [ ]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [1]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [ ]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = 
)PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [ ]:
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chain=destination_chain,
    default_chain=default_chain,
    verbose=True
)

In [ ]:
chain.run("What is black body radiation?")

**Output** 'Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation and emits radiation at all frequencies. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This type of radiation is important in understanding concepts such as thermal radiation and the behavior of objects at different temperatures.'

In [ ]:
chain.run("Why does every cell in our body contain DNA?")

**Output** : 'Every cell in our body contains DNA because DNA carries the genetic information that determines the characteristics and functions of an organism. DNA contains the instructions for building and maintaining an organism, including the proteins that are essential for cell function and structure. This genetic information is passed down from parent to offspring and is essential for the growth, development, and functioning of all cells in the body. Having DNA in every cell ensures that the genetic information is preserved and can be used to carry out the necessary processes for life.'|